In [84]:
import numpy as np
import pandas as pd

from src.util.model_utils import read_jsonl

%cd /Users/amir/Data/Projects/SQL_parser

def find_starts(d):
    starts = [0]
    for i in range(1, len(d) - 1):
        prev = d.iloc[i - 1]
        cur = d.iloc[i]
        nxt = d.iloc[i + 1]
        if prev['ct'] > cur['ct']:
            starts.append(cur['ts'])
    return starts


def proc_df(path: str):
    data = read_jsonl(path)
    rows = []
    for row in data:
        ts = row['record']['time']['timestamp']
        ts = int(ts)
        util = row['record']['extra']['util']
        rows.append({"ts": ts, 'ct': util['cpu_time'], 'cp': util['cpu'], 'mem': util['mem']})
    df = pd.DataFrame(rows)
    df['ts'] = df['ts'] - df['ts'].min()
    df['cc'] = df['cp'] / 100
    df['ct'] = df['ct'].apply(np.ceil)
    df['ctd'] = df['ct'].diff()
    df['ctd'] = df['ctd'].clip(lower=0)
    return df






/Users/amir/Data/Projects/SQL_parser


In [94]:
def get_util_stats(model):
    df = proc_df(f"data/{model}_bird_logs/util.jsonl")
    timing_df = pd.read_csv(f"charts/{model}_timing.csv")
    starts = find_starts(df)
    stats = dict()
    pre = timing_df.groupby(["tmp","itr"]).sum()
    cts = df[df['ts'].isin(starts)]['ct']
    stats['Total CPU Time'] = cts.sum() / 9
    stats['Max Mem'] = df['mem'].max()
    stats['Max cores'] = df['cc'].max()
    stats['Max cpu percentage'] = df['cp'].max()
    stats['Total Time'] = pre['time'].mean()
    return stats



In [95]:
stats = get_util_stats("din")
print(stats)


{'Total CPU Time': 945.1111111111111, 'Max Mem': 26895.74609375, 'Max cores': 211.88100000000003, 'Max cpu percentage': 21188.100000000002, 'Total Time': 23229.333333333332}
